[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/process_tomography/process_tomography.ipynb)

# Quantum Process Tomography

Quantum process tomography recovers a quantum device's channel on every qubit state from outcome probabilities measured on known input states. A qubit state is modelled as a point inside a ball in three-dimensional projective space, where pure states form the boundary sphere and mixed states fill the interior around the completely mixed centre. Because every outcome probability depends affinely on the state, each detector outcome is modelled as a plane, and four input states arranged as a regular tetrahedron span the ball and determine the channel everywhere.

Each state carries unit weight read by the plane at infinity as `w & state`, and a detector plane `effect` reads its outcome probability directly through the pairing `effect & state`. Coherent turns act by rotors around the centre, while dephasing and relaxation toward the north pole remain linear maps `Point <- Point` because the weight term `north * (w & Point)` shifts the ball's centre without a separate translation vector. Reconstruction requires no metric inner product because each outcome's dual point is the meet of the other three outcome planes and each preparation's dual plane is the join of the other three preparations. See [the Bloch-vector twin notebook](process_tomography_bloch.ipynb) for the conventional Pauli-algebra formulation, following [Chuang and Nielsen](https://arxiv.org/abs/quant-ph/9610001) and the symmetric tetrahedral measurement of [Renes et al.](https://arxiv.org/abs/quant-ph/0310075).

In density-matrix and Pauli-transfer-matrix notation, a state is expanded as $\rho = \tfrac12(I + \mathbf r \cdot \boldsymbol\sigma)$, a channel acts either by Kraus matrices $\mathcal E(\rho) = \sum_k K_k \rho K_k^\dagger$ or by the real $4 \times 4$ matrix $R_{ij} = \tfrac12\operatorname{tr}[\sigma_i\,\mathcal E(\sigma_j)]$ on column vectors $(1, r_x, r_y, r_z)^\mathsf{T}$, and reconstruction inverts the $4 \times 4$ coordinate matrices of the prepared states and POVM elements $F_b = \tfrac14(I + \mathbf n_b \cdot \boldsymbol\sigma)$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D
from examples.quantum.process_tomography import render
from examples.util.animation import show_states

np.set_printoptions(precision=4, suppress=True)

ga = PGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Bivector = ga.gatype.bivector()                                              # Bivector+
# A state is a unit-weight point of the Bloch ball; an effect is a plane that reads states.
Point = ga.gatype.antivector()                                               # Antivector-
Effect = ga.gatype.vector()                                                  # Vector+
Channel = ga.gatype((Point, Point))                                          # Point- <- Point-

# The centre of the ball (the completely mixed state) and the north pole toward which relaxation pulls.
centre = mv.point([0, 0, 0])                                                 # [] Point-
north = mv.point([0, 0, 1])                                                  # [] Point-
# Indices of the complementary triple for each of four spanning preparations or outcomes.
other_indices = np.array([[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]])

## Preparing, evolving, and measuring

Four pure states at the corners of a regular tetrahedron pass through each gate, and a four-outcome tetrahedral detector measures each outgoing state. Combining the plane at infinity with four analyzer planes facing the tetrahedron's directions gives four effects that add up to `w`, so the four outcome probabilities add up to the state's unit weight. Dephasing mixes a state with its half turn `xy >> state`, while unobserved relaxation toward `north` retains transverse coherence amplitude `sqrt(1 - loss)`, the quantum ceiling derived from operator sandwiches in [the Bloch-vector twin notebook](process_tomography_bloch.ipynb).

In density-matrix notation, the POVM elements read as $F_b = \tfrac14(I + \mathbf n_b \cdot \boldsymbol\sigma)$ and the Born probabilities as $p(b \mid a) = \operatorname{tr}[F_b\,\mathcal E(\rho_a)]$.

In [ ]:
labels = ("Rotation", "Rotation + dephasing", "Rotation + relaxation")
angles = np.array([0.075, 0.075, 0.075])
phase_flip = np.array([0.0, 0.06, 0.0])
loss = np.array([0.0, 0.0, 0.10])
rotation_plane = (mv.yz + mv.zx + mv.xy) / np.sqrt(3)                        # [] Bivector+

tetrahedron = mv.direction([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]) / np.sqrt(3)       # [4] Point-
analyzers = mv.vector([[1, 1, 1, 0], [1, -1, -1, 0], [-1, 1, -1, 0], [-1, -1, 1, 0]]) / np.sqrt(3)  # [4] Effect+
preparations = centre + tetrahedron                                          # [4] Point-
effects = (mv.w + analyzers) / 4                                             # [4] Effect+

In [ ]:
def noisy_gate(
    angles: np.ndarray, phase_flip: np.ndarray, loss: np.ndarray, rotation_plane: Bivector,
) -> Channel:
    """Independent phase noise, relaxation toward the north pole, and a coherent turn form a gate channel (`[...] Point- <- Point-`)."""
    # Phase flip replaces the state with its half turn about the z axis with probability phase_flip.
    dephasing = (1 - phase_flip) * Point + phase_flip * (mv.xy >> Point)     # [...] Point- <- Point-
    # Relaxation sends decayed weight to the north pole, read off by pairing with the plane at infinity.
    decay = north * (mv.w & Point)                                           # [] Point- <- Point-
    # A half turn about the z axis reverses the transverse spin components while fixing the axis.
    transverse = (Point - (mv.xy >> Point)) / 2                              # [] Point- <- Point-
    # Unobserved decay retains transverse coherence amplitude sqrt(1 - loss) rather than classical 1 - loss.
    damping = (1 - loss) * Point + loss * decay + (np.sqrt(1 - loss) - (1 - loss)) * transverse  # [...] Point- <- Point-
    rotations = (rotation_plane * (-angles / 2)).exp()                       # [...] Rotor+
    return rotations >> damping(dephasing)                                   # [...] Point- <- Point-


def outcome_probabilities(channel: Channel, states: Point, effects: Effect) -> Scalar:
    """Pairing `effects` against mapped `states` gives the Born outcome probabilities (`[...] Scalar+`)."""
    outputs = channel[..., None](states)                                     # [...] Point-
    return effects & outputs[..., None]                                      # [...] Scalar+


true_channels = noisy_gate(angles, phase_flip, loss, rotation_plane)         # [3] Point- <- Point-
measurements = outcome_probabilities(true_channels, preparations, effects)   # [3, 4, 4] Scalar+
render.experiment_figure(preparations, measurements, labels, render.ball_positions);

The left panel places the four tetrahedral preparations `A`, `B`, `C`, and `D` inside the Bloch sphere, and the three log-scaled heatmaps show the sixteen outcome probabilities for pure rotation, dephasing, and relaxation. Diagonal entries stay brightest because a small turn keeps each state closest to its matching analyzer, while noise raises the off-diagonal probabilities.

## Recovering the channel

Three detector planes meet at a dual point where all three read zero, and scaling that meet so the fourth plane reads one isolates the fourth outcome's contribution to each outgoing state. Symmetrically, joining three preparation points forms a dual plane that vanishes on all three, and scaling it to read one on the fourth isolates that preparation's weight in any input state. Summing the dyads `recovered_outputs * (preparation_dual & Point)` builds the reconstructed `Point <- Point` map from meets and joins alone.

In matrix notation, the dual preparation planes correspond to the rows of $P^{-1}$ for the $4 \times 4$ matrix $P$ whose columns hold the homogeneous coordinates of the four prepared states.

In [ ]:
n_latitudes = 12
n_longitudes = 2 * n_latitudes

In [ ]:
def reconstructed_channel(preparations: Point, effects: Effect, measurements: Scalar) -> Channel:
    """Reciprocal meets of `effects` and joins of `preparations` recover the linear channel (`[...] Point- <- Point-`)."""
    # Each outcome's dual point lies on the meet of the other three effect planes, scaled so its own effect reads one.
    meets = effects[other_indices[:, 0]] ^ effects[other_indices[:, 1]] ^ effects[other_indices[:, 2]]  # [4] Antivector+
    measurement_dual = meets / (effects & meets)                             # [4] Point-
    recovered_outputs = (measurements * measurement_dual).sum(axis=-1)       # [..., 4] Point-
    # Each preparation's dual plane joins the other three preparations, scaled to read one on its own preparation.
    joins = preparations[other_indices[:, 0]] & preparations[other_indices[:, 1]] & preparations[other_indices[:, 2]]  # [4] Vector-
    preparation_dual = joins / (joins & preparations)                        # [4] Effect+
    return (recovered_outputs * (preparation_dual & Point)).sum(axis=-1)     # [...] Point- <- Point-


def pure_sphere(n_latitudes: int, n_longitudes: int) -> Point:
    """Inclination tilts and azimuthal turns sweep `north` across a closed sphere grid (`[n_latitudes + 1, n_longitudes] Point-`)."""
    inclinations = np.linspace(0, np.pi, n_latitudes + 1)
    azimuths = np.linspace(0, 2 * np.pi, n_longitudes, endpoint=False)
    tilts = (mv.zx * (-inclinations / 2)).exp()                              # [n_latitudes + 1] Rotor+
    turns = (mv.xy * (-azimuths / 2)).exp()                                  # [n_longitudes] Rotor+
    meridian = tilts >> north                                                # [n_latitudes + 1] Point-
    return turns[None, :] >> meridian[:, None]                               # [n_latitudes + 1, n_longitudes] Point-


learned_channels = reconstructed_channel(preparations, effects, measurements)  # [3] Point- <- Point-
surface = pure_sphere(n_latitudes, n_longitudes)                             # [n_latitudes + 1, n_longitudes] Point-
sphere_images = learned_channels[:, None, None](surface)                     # [3, n_latitudes + 1, n_longitudes] Point-
probe_images = learned_channels[:, None](preparations)                       # [3, 4] Point-
render.channels_figure(surface, sphere_images, preparations, probe_images, labels, render.ball_positions);

Each panel overlays the translucent unit sphere of pure inputs and the four larger translucent prepared probes with the reconstructed output surface and smaller solid output probes. Pure rotation maps the sphere onto itself, dephasing pinches the equator inward toward the `z` axis while fixing the poles, and relaxation both contracts the sphere and pulls its centre toward the north pole.

## Predicting unseen inputs

Because outcome probabilities depend linearly on the projective state, the reconstructed channel predicts detector probabilities across pure and mixed states absent from the calibration set. Under relaxation, the weight term `north * (w & Point)` shifts even the completely mixed state at the centre toward the north pole, whereas dephasing leaves the centre fixed.

In Bloch-vector notation, the channel acts as the affine map $\mathbf r' = A\mathbf r + \mathbf t$, separating the linear contraction $A$ from the centre shift $\mathbf t$.

In [ ]:
validation_directions = mv.direction(
    [[1, 0, 0], [0, 1, 0], [0, 0, 1], [-1, 0, 0], [0, -1, 0], [0, 0, -1]],
)                                                                            # [6] Point-
bloch_lengths = np.array([1.0, 0.7, 0.3, 1.0, 0.7, 0.0])

In [ ]:
validation_states = centre + validation_directions * bloch_lengths           # [6] Point-
observed_probabilities = outcome_probabilities(true_channels, validation_states, effects)      # [3, 6, 4] Scalar+
predicted_probabilities = outcome_probabilities(learned_channels, validation_states, effects)  # [3, 6, 4] Scalar+
render.predictions_figure(observed_probabilities, predicted_probabilities, labels);

All twenty-four state-outcome probabilities for each of the three channels lie along the diagonal equality line, confirming exact recovery across pure, partially mixed, and completely mixed validation states.

## Spanning preparations

Reciprocal dual planes exist whenever the four preparations span three-dimensional projective space, so their four-way join `preparations[0] & preparations[1] & preparations[2] & preparations[3]` is a nonzero volume antiscalar. Two opposite states such as `north` and `south` join only to the axial line `north & south`, leaving turns around that line and contractions across it invisible to the calibration.

In [ ]:
south = mv.point([0, 0, -1])                                                 # [] Point-
tetrahedral_span = preparations[0] & preparations[1] & preparations[2] & preparations[3]  # [] Antiscalar-
opposite_span = north & south                                                # [] Antibivector-
print("Four tetrahedral probes join to a volume:\n", tetrahedral_span.coefficient_table())
print("Two opposite probes join to a line:\n", opposite_span.coefficient_table())

## Predicting repeated uses

Composing the reconstructed `Point <- Point` map with itself predicts any number of successive passes through the gate without further calibration measurements.

In Pauli-transfer-matrix notation, $n$ successive applications of the gate read as the matrix power $R^n$.

In [ ]:
n_uses = 48
frame_duration_ms = 40

In [ ]:
def channel_powers(channel: Channel, n_uses: int) -> Iterator[Channel]:
    """Yields the composed channels (`[...] Point- <- Point-`) for one through `n_uses` applications of `channel`."""
    channel_power = channel                                                  # [...] Point- <- Point-
    for _ in range(n_uses):
        yield channel_power
        channel_power = channel(channel_power)                               # [...] Point- <- Point-


frames = (
    (channel_power[:, None, None](surface), channel_power[:, None](preparations))
    for channel_power in channel_powers(learned_channels, n_uses)
)
power_frames = show_states(
    frames, render.channel_animation, surface, preparations, labels, render.ball_positions, render.panel_size,
    name="process_tomography", duration_ms=frame_duration_ms,
);

Across repeated passes, pure rotation winds the four solid output probes steadily around the `(1, 1, 1)` axis on the unit sphere, dephasing spirals the ellipsoid inward toward the tilted rotation axis near the centre, and relaxation draws the contracting ellipsoid toward a fixed point near the north pole.

In [ ]:
# checks
assert (measurements >= -1e-12).all()
assert ((measurements.sum(axis=-1) - 1).abs() < 1e-12).all()
assert ((outcome_probabilities(learned_channels, preparations, effects) - measurements).abs() < 1e-12).all()
assert ((observed_probabilities - predicted_probabilities).abs() < 1e-12).all()
assert tetrahedral_span.norm() > 1.0
assert (opposite_span & centre).norm() < 1e-12
assert ((opposite_span & preparations).norm() > 0.1).all()
assert len(power_frames) == n_uses
final_spheres, final_probes = power_frames[-1]
assert (((mv.w & final_probes) - 1).abs() < 1e-10).all()
assert (((final_spheres[0] - centre).antinorm() - 1).abs() < 1e-10).all()
assert ((final_spheres[1] - centre).antinorm() < 0.4).all()
assert ((final_spheres[2] - centre).antinorm() < 0.99).all()
assert ((final_spheres[2] - north).antinorm() < (final_spheres[2] - south).antinorm()).all()